# Mat & Comp – ML 9: Regresión múltiple y polinomial
## Más características, curvas y el arte de frenar

**Curso:** Machine learning (Sección 3) · **Duración sugerida:** 2 sesiones de 50 minutos · **Requisito:** ML 8

### Objetivos de aprendizaje
Al terminar esta lección podrás:
1. Escribir la regresión múltiple como un **sistema de matrices** y resolverla con numpy.
2. Comparar la importancia de las características **normalizándolas**, y convertir columnas de texto en números.
3. Ajustar **curvas** con un modelo lineal creando características nuevas, como $x^2$.
4. Reconocer el sobreajuste por **exceso de características** y controlarlo con **regularización** (Ridge y Lasso).
5. Elegir la fuerza de la regularización con **validación cruzada** (ML 7).

## 🔍 ¿El doble de rápido, el doble de distancia?

Un carro a **50 km/h** necesita unos **28 metros** para detenerse por completo (contando el tiempo que tarda el conductor en reaccionar).

✏️ *¿Cuántos metros crees que necesita a **100 km/h**? ¿El doble, 56 metros? ¿Más? ¿Menos?*

**Mi respuesta:**

Lo descubriremos con datos. Para eso necesitamos modelos que no solo dibujen rectas: hoy aprenderemos a usar **varias características** a la vez, a ajustar **curvas** con las mismas herramientas de ML 2, y a evitar que tantas características terminen en sobreajuste.

## 1. Regresión múltiple, ahora con matrices

En ML 2 predijimos la nota con varias columnas usando `LinearRegression`. Veamos qué hace por dentro. Usaremos cinco características: horas de estudio, de sueño y de redes, el grado y si practica deporte.

La columna `deporte` dice «sí» o «no». Los modelos solo entienden números, así que la convertimos en **1** (sí) y **0** (no).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

URL = "https://raw.githubusercontent.com/emersonjleon/compumath/main/lecciones/seccion3_ml/datos/estudiantes.csv"
datos = pd.read_csv(URL)
datos["horas_sueno"] = datos["horas_sueno"].fillna(datos["horas_sueno"].mean())   # como en ML 1 y ML 2
datos["deporte"] = (datos["deporte"] == "sí").astype(int)                          # sí → 1, no → 0

columnas = ["horas_estudio", "horas_sueno", "horas_redes", "grado", "deporte"]
X = datos[columnas].to_numpy()
y = datos["nota_matematicas"].to_numpy()
print("X tiene forma", X.shape, "  y tiene forma", y.shape)
datos[columnas + ["nota_matematicas"]].head()

El modelo es

$$\hat{y} = w_1 x_1 + w_2 x_2 + w_3 x_3 + w_4 x_4 + w_5 x_5 + b$$

Si agregamos a `X` una columna de **unos** (para el término $b$), las 60 predicciones se calculan de una vez con el producto de matrices de Python 11:

$$\hat{y} = A\,\theta, \qquad A = [\,X \;|\; 1\,], \qquad \theta = (w_1, \dots, w_5, b)$$

Los **mínimos cuadrados** buscan el $\theta$ que hace más pequeño el error cuadrático. numpy lo resuelve con `np.linalg.lstsq` («least squares»):

In [ ]:
A = np.column_stack([X, np.ones(len(X))])     # agregar la columna de unos
theta, *_ = np.linalg.lstsq(A, y, rcond=None)

for nombre, valor in zip(columnas + ["b (intercepto)"], theta):
    print(f"{nombre:15} {valor:+.4f}")

In [ ]:
from sklearn.linear_model import LinearRegression

lineal = LinearRegression().fit(X, y)
print("scikit-learn:", lineal.coef_.round(4), round(lineal.intercept_, 4))
print("numpy:       ", theta[:-1].round(4), round(theta[-1], 4))
print(f"R² = {lineal.score(X, y):.3f}")

¡Iguales! `LinearRegression` no es más que mínimos cuadrados con una columna de unos.

### ¿Qué característica importa más?

Cada coeficiente dice cuánto cambia la nota si esa característica sube **1 unidad** y las demás se quedan quietas. Pero las unidades son distintas: 1 hora de estudio a la semana no es comparable con «1 grado escolar» o con «pasar de no hacer deporte a hacerlo».

Para comparar, **normalizamos** cada columna (como en ML 4): le restamos su promedio y la dividimos por su desviación estándar. Así, cada coeficiente dice cuánto cambia la nota si esa característica sube **una desviación estándar**.

In [ ]:
from sklearn.preprocessing import StandardScaler

Xn = StandardScaler().fit_transform(X)
lineal_n = LinearRegression().fit(Xn, y)

orden = np.argsort(np.abs(lineal_n.coef_))
plt.figure(figsize=(6, 3))
plt.barh(np.array(columnas)[orden], lineal_n.coef_[orden],
         color=["seagreen" if c > 0 else "crimson" for c in lineal_n.coef_[orden]])
plt.axvline(0, color="k", lw=1)
plt.xlabel("cambio en la nota por cada desviación estándar")
plt.title("Coeficientes normalizados")
plt.show()

✏️ *¿Cuál característica pesa más? ¿Alguna tiene un efecto negativo? ¿Alguna casi no influye?*

Recuerda (ML 1): que un coeficiente sea grande **no** prueba que esa característica **cause** la nota. Solo dice que, en estos datos, van juntas.

## 2. Curvas con un modelo lineal: la distancia de frenado

Volvamos a la pregunta inicial. Estos son datos de 30 pruebas de frenado (simulados con fórmulas de la física y algo de ruido, porque cada carro, conductor y pavimento es distinto):

In [ ]:
rng = np.random.default_rng(1)
v = np.sort(rng.uniform(20, 110, 30)).round(0)             # velocidad en km/h
d = (0.28 * v + v ** 2 / 178 + rng.normal(0, 5, 30)).round(1)   # distancia de frenado en metros

plt.figure(figsize=(6.5, 3.5))
plt.scatter(v, d, color="black", s=25)
plt.xlabel("velocidad (km/h)"); plt.ylabel("distancia de frenado (m)")
plt.title("30 pruebas de frenado")
plt.grid(alpha=0.3)
plt.show()

Una recta no captura bien la curvatura. Pero hay un truco: si creamos una **nueva característica**, $v^2$, el modelo

$$\hat{d} = w_1 v + w_2 v^2 + b$$

sigue siendo una regresión **múltiple**: es lineal en los coeficientes $w_1$, $w_2$, $b$, aunque dibuje una curva. ¡Podemos resolverlo con los mismos mínimos cuadrados!

In [ ]:
# Recta: columnas [v, 1]
A1 = np.column_stack([v, np.ones(len(v))])
th1, *_ = np.linalg.lstsq(A1, d, rcond=None)

# Parábola: columnas [v, v², 1]
A2 = np.column_stack([v, v ** 2, np.ones(len(v))])
th2, *_ = np.linalg.lstsq(A2, d, rcond=None)

def ecm(real, pred):
    return np.mean((real - pred) ** 2)

print(f"Recta:    d = {th1[0]:.3f} v + {th1[1]:.2f}              ECM = {ecm(d, A1 @ th1):.1f}")
print(f"Parábola: d = {th2[0]:.3f} v + {th2[1]:.5f} v² + {th2[2]:.2f}   ECM = {ecm(d, A2 @ th2):.1f}")

vs = np.linspace(0, 130, 200)
plt.figure(figsize=(6.5, 3.5))
plt.scatter(v, d, color="black", s=25)
plt.plot(vs, th1[0] * vs + th1[1], label="recta")
plt.plot(vs, th2[0] * vs + th2[1] * vs ** 2 + th2[2], label="parábola")
plt.xlabel("velocidad (km/h)"); plt.ylabel("distancia (m)")
plt.legend(); plt.grid(alpha=0.3)
plt.show()

In [ ]:
for vel in [50, 100]:
    print(f"A {vel} km/h la parábola predice {th2[0] * vel + th2[1] * vel ** 2 + th2[2]:.0f} m")

A 100 km/h la distancia es **unas tres veces** la de 50 km/h, no el doble. La parte de reacción crece en proporción a la velocidad, pero la de frenado crece con su **cuadrado**: el doble de velocidad, cuatro veces la distancia de frenado. 🚗💨

### Con scikit-learn: `PolynomialFeatures`

`PolynomialFeatures(degree=2)` crea las columnas $v$ y $v^2$ por nosotros, y con un `Pipeline` (ML 4) se encadena con la regresión:

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

print(PolynomialFeatures(degree=2, include_bias=False).fit_transform([[3], [10]]))

parabola = make_pipeline(PolynomialFeatures(degree=2, include_bias=False), LinearRegression())
parabola.fit(v.reshape(-1, 1), d)
print("Coeficientes:", parabola[-1].coef_, "  intercepto:", round(parabola[-1].intercept_, 2))

Con varias columnas, `PolynomialFeatures` también crea los **productos** entre ellas. Con $x_1$, $x_2$ y grado 2: $x_1, x_2, x_1^2, x_1 x_2, x_2^2$.

## 3. ¿Y si agregamos muchas potencias?

Si $v^2$ ayudó, ¿por qué no $v^3, v^4, \dots, v^{12}$? Ya en ML 7 vimos que un polinomio de grado alto **memoriza el ruido**. Veámoslo otra vez, ahora mirando los **coeficientes**. (Dividimos la velocidad por 100 para que las potencias no se vuelvan números gigantes.)

In [ ]:
from sklearn.model_selection import cross_val_score, KFold

V = (v / 100).reshape(-1, 1)
grupos = KFold(5, shuffle=True, random_state=0)

def modelo_poli(grado):
    return make_pipeline(PolynomialFeatures(grado, include_bias=False), StandardScaler(), LinearRegression())

for grado in [1, 2, 4, 8, 12]:
    m = modelo_poli(grado).fit(V, d)
    error_cv = -cross_val_score(modelo_poli(grado), V, d, cv=grupos, scoring="neg_mean_squared_error").mean()
    mayor = np.abs(m[-1].coef_).max()
    print(f"grado {grado:2}: ECM entrenamiento = {ecm(d, m.predict(V)):6.1f}   ECM validación cruzada = {error_cv:9.1f}   coeficiente más grande = {mayor:10.1f}")

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(12, 3.5))
vs = np.linspace(15, 115, 300)
for grado in [2, 12]:
    m = modelo_poli(grado).fit(V, d)
    ejes[0].plot(vs, m.predict((vs / 100).reshape(-1, 1)), label=f"grado {grado}")
ejes[0].scatter(v, d, color="black", s=20)
ejes[0].set_ylim(-10, 140)
ejes[0].set_xlabel("velocidad (km/h)"); ejes[0].set_ylabel("distancia (m)")
ejes[0].legend(); ejes[0].grid(alpha=0.3)

m12 = modelo_poli(12).fit(V, d)
ejes[1].bar(range(1, 13), m12[-1].coef_, color="gray")
ejes[1].set_xlabel("potencia de v"); ejes[1].set_title("Coeficientes del grado 12")
ejes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

Con grado 12, el error de **entrenamiento** baja, pero el de **validación cruzada** sube: sobreajuste. Y fíjate en los coeficientes: son **enormes** y de signos alternados. El modelo suma y resta números gigantes que casi se cancelan para pasar cerca de cada punto. Una pequeña diferencia en los datos cambia todo.

¿Podemos usar muchas características **sin** que los coeficientes se disparen?

## 4. Regularización: castigar los coeficientes grandes

La idea de la **regresión Ridge** es cambiar lo que minimizamos. Además del error, sumamos un castigo por tener coeficientes grandes:

$$\text{costo} = \underbrace{\sum (y - \hat{y})^2}_{\text{error}} \; + \; \lambda \underbrace{(w_1^2 + w_2^2 + \cdots)}_{\text{tamaño de los pesos}}$$

El número $\lambda$ (lambda) controla la fuerza del castigo:
- $\lambda = 0$: mínimos cuadrados normales;
- $\lambda$ pequeño: frena un poco los coeficientes;
- $\lambda$ enorme: los aplasta hacia 0 y el modelo se vuelve casi una línea horizontal.

Con cálculo se demuestra que la solución tiene una fórmula exacta, muy parecida a la de mínimos cuadrados (con las columnas normalizadas y $y$ centrada):

$$w = (X^T X + \lambda I)^{-1} X^T y$$

Programémosla con numpy para el polinomio de grado 12:

In [ ]:
P = PolynomialFeatures(12, include_bias=False).fit_transform(V)
escala = StandardScaler().fit(P)
Pn = escala.transform(P)
y_centro = d.mean()

def ridge_a_mano(Xn, y, lam):
    n_col = Xn.shape[1]
    w = np.linalg.solve(Xn.T @ Xn + lam * np.eye(n_col), Xn.T @ (y - y.mean()))
    return w

vs = np.linspace(15, 115, 300)
Pvs = escala.transform(PolynomialFeatures(12, include_bias=False).fit_transform((vs / 100).reshape(-1, 1)))

fig, ejes = plt.subplots(1, 2, figsize=(12, 3.5))
ejes[0].scatter(v, d, color="black", s=20)
for lam in [0.0001, 0.1, 10, 1000]:
    w = ridge_a_mano(Pn, d, lam)
    ejes[0].plot(vs, Pvs @ w + y_centro, label=f"λ = {lam}")
    print(f"λ = {lam:7}: coeficiente más grande = {np.abs(w).max():8.2f}   ECM entrenamiento = {ecm(d, Pn @ w + y_centro):6.1f}")
ejes[0].set_ylim(-10, 140)
ejes[0].set_xlabel("velocidad (km/h)"); ejes[0].set_ylabel("distancia (m)")
ejes[0].legend(); ejes[0].grid(alpha=0.3)

ejes[1].bar(range(1, 13), ridge_a_mano(Pn, d, 0.1), color="tab:orange")
ejes[1].set_xlabel("potencia de v"); ejes[1].set_title("Coeficientes con λ = 0.1")
ejes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

Con un castigo moderado, el polinomio de grado 12 se **calma**: la curva se parece a la parábola, aunque tenga 12 características. Con $\lambda$ demasiado grande, el modelo no puede ni seguir la tendencia (subajuste, como en ML 7).

### Con scikit-learn: `Ridge`

En scikit-learn, $\lambda$ se llama `alpha`. Comprobemos que da lo mismo que nuestra fórmula:

In [ ]:
from sklearn.linear_model import Ridge

r = Ridge(alpha=0.1).fit(Pn, d)
print("scikit-learn:", r.coef_[:4].round(3), "...")
print("a mano:      ", ridge_a_mano(Pn, d, 0.1)[:4].round(3), "...")

### ¿Qué λ elegir? Validación cruzada

$\lambda$ no se aprende de los datos de entrenamiento (con ellos, siempre ganaría $\lambda = 0$). Se elige con **validación cruzada**, como en ML 7:

In [ ]:
alphas = 10.0 ** np.arange(-5, 4)
errores = []
for a in alphas:
    m = make_pipeline(PolynomialFeatures(12, include_bias=False), StandardScaler(), Ridge(alpha=a))
    errores.append(-cross_val_score(m, V, d, cv=grupos, scoring="neg_mean_squared_error").mean())

mejor = alphas[np.argmin(errores)]
plt.figure(figsize=(6, 3.2))
plt.plot(alphas, errores, "o-")
plt.xscale("log"); plt.yscale("log")
plt.xlabel("alpha (λ), escala logarítmica"); plt.ylabel("ECM de validación cruzada (escala log.)")
plt.title(f"El mejor λ es {mejor:g}")
plt.grid(alpha=0.3)
plt.show()

parabola_cv = -cross_val_score(make_pipeline(PolynomialFeatures(2, include_bias=False), LinearRegression()),
                               V, d, cv=grupos, scoring="neg_mean_squared_error").mean()
print(f"Grado 12 con el mejor λ: ECM = {min(errores):.1f}")
print(f"Parábola sin regularizar: ECM = {parabola_cv:.1f}")

Otra vez la **curva en U** de ML 7: a la izquierda (λ pequeño) sobreajuste, a la derecha (λ grande) subajuste. Con el mejor λ, el polinomio de grado 12 queda casi tan bien como la parábola, aunque nadie le dijo que la física era cuadrática.

💡 **Conexión con ML 8:** la regresión logística de scikit-learn ya usaba regularización. Su parámetro `C` es lo contrario de λ: `C` grande es castigo pequeño. Por eso usamos `C=1e6` para que coincidiera con nuestro cálculo a mano.

## 5. Lasso: regularización que elige características

Volvamos a los estudiantes. Supongamos que alguien agrega a la tabla **30 columnas de puro ruido** (números al azar que no tienen nada que ver con la nota). Un buen modelo debería ignorarlas.

In [ ]:
rng2 = np.random.default_rng(0)
ruido = rng2.normal(size=(len(X), 30))
X_mas = np.column_stack([X, ruido])
nombres = columnas + [f"ruido{i}" for i in range(1, 31)]
print("Ahora hay", X_mas.shape[1], "características para", X_mas.shape[0], "estudiantes")

La regresión **Lasso** usa otro castigo: la suma de los **valores absolutos** de los coeficientes, $\lambda(|w_1| + |w_2| + \cdots)$. Este castigo tiene una propiedad muy útil: empuja muchos coeficientes a **exactamente cero**. Es decir, **elige** qué características usar.

In [ ]:
from sklearn.linear_model import Lasso

modelos = {
    "lineal (sin castigo)": make_pipeline(StandardScaler(), LinearRegression()),
    "Ridge (alpha = 10)": make_pipeline(StandardScaler(), Ridge(alpha=10)),
    "Lasso (alpha = 0.05)": make_pipeline(StandardScaler(), Lasso(alpha=0.05)),
}
grupos_e = KFold(5, shuffle=True, random_state=0)
print("R² con validación cruzada (más alto es mejor):")
print(f"  {'solo las 5 columnas reales':22}: {cross_val_score(LinearRegression(), X, y, cv=grupos_e).mean():.3f}")
for nombre, m in modelos.items():
    print(f"  {nombre:22}: {cross_val_score(m, X_mas, y, cv=grupos_e).mean():.3f}")

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(12, 8), sharey=True)
for eje, (nombre, m) in zip(ejes, [("lineal (sin castigo)", modelos["lineal (sin castigo)"]),
                                   ("Lasso (alpha = 0.05)", modelos["Lasso (alpha = 0.05)"])]):
    m.fit(X_mas, y)
    coef = m[-1].coef_
    eje.barh(nombres, coef, color=["tab:blue"] * 5 + ["gray"] * 30)
    eje.axvline(0, color="k", lw=1)
    eje.set_title(nombre)
    eje.invert_yaxis()
plt.tight_layout()
plt.show()

lasso = modelos["Lasso (alpha = 0.05)"]
print("Columnas que Lasso dejó con coeficiente distinto de cero:")
print([n for n, c in zip(nombres, lasso[-1].coef_) if c != 0])

Con 35 columnas para solo 60 estudiantes, el modelo sin castigo le da peso a muchas columnas de ruido: aprende casualidades de estos estudiantes y predice muy mal con datos nuevos. Ridge ayuda, pero sigue usando todas las columnas. Lasso pone en cero **todas** las columnas de ruido (y también `grado` y `deporte`, que casi no aportaban) y se queda con las características que importan.

| Modelo | Castigo | Efecto |
|---|---|---|
| Lineal | ninguno | puede sobreajustar con muchas características |
| Ridge | $\lambda \sum w^2$ | encoge todos los coeficientes, ninguno llega a 0 |
| Lasso | $\lambda \sum \lvert w \rvert$ | encoge y pone muchos en 0: elige características |

## 6. Errores comunes

**Error 1: pasar una lista simple a `PolynomialFeatures`.** Como todo en scikit-learn, espera una tabla:

In [ ]:
PolynomialFeatures(degree=2).fit_transform(v)

La solución es `v.reshape(-1, 1)`.

**Error 2: regularizar sin normalizar.** El castigo suma los coeficientes de todas las columnas, pero si una columna está en kilómetros y otra en milímetros, sus coeficientes tienen tamaños muy distintos y el castigo es injusto. Pon siempre `StandardScaler()` antes de `Ridge` o `Lasso`.

**Error 3: extrapolar con polinomios.** Fuera del rango de los datos, un polinomio de grado alto se dispara. Prueba predecir la distancia a 150 km/h con el grado 12 sin regularizar.

**Error 4: demasiadas columnas sin darse cuenta.** Con 10 características y grado 3, `PolynomialFeatures` crea 285 columnas. Con 60 estudiantes, eso es una receta para el sobreajuste.

**Error 5: elegir λ mirando los datos de prueba.** λ se elige con validación cruzada sobre el entrenamiento; la prueba se usa una sola vez, al final (ML 7).

## 🤔 ¿Qué pasa si…?

Usa la celda de abajo. Antes de ejecutar, **predice** el resultado.

1. …en la sección 1 quitas la columna de unos de `A`? ¿Cambian los coeficientes? ¿Y el error?
2. …ajustas a los datos de frenado un polinomio de grado 3 sin regularizar? ¿Cuánto vale el coeficiente de $v^3$?
3. …usas `Lasso(alpha=1)` en la sección 5? ¿Cuántas columnas sobreviven? ¿Y con `alpha=0.001`?
4. …calculas `PolynomialFeatures(2, include_bias=False).fit_transform([[2, 3]])`? ¿Qué es cada número?
5. …predices la distancia a 150 km/h con la parábola y con el grado 12 sin regularizar (`modelo_poli(12)`)? ¿Cuál le crees?

In [ ]:
# Prueba aquí tus ideas

## ✍️ Ejercicios

**Ejercicio 1 (fácil). Leer el modelo.** Con los coeficientes `theta` de la sección 1 (sin normalizar), predice la nota de una estudiante de grado 10 que estudia 8 horas a la semana, duerme 7 horas, usa redes 2 horas al día y practica deporte. Compruébalo con `lineal.predict`.

**Ejercicio 2 (fácil). Contar columnas.** Sin usar Python, ¿cuántas columnas crea `PolynomialFeatures(degree=2, include_bias=False)` con 3 características? Escríbelas ($x_1, x_2, \dots$). Comprueba con Python. ¿Y con grado 3?

**Ejercicio 3 (medio). Solo reacción y solo frenado.** Ajusta a los datos de frenado el modelo $\hat{d} = w_1 v + w_2 v^2$ **sin intercepto** (pista: `np.linalg.lstsq` sin la columna de unos, o `LinearRegression(fit_intercept=False)`). Compara $w_1$ con 0.28 (el tiempo de reacción de 1 s convertido) y $w_2$ con $1/178 \approx 0.0056$. ¿Recuperaste la física?

**Ejercicio 4 (medio). Curva de Ridge.** Para la regresión de la sección 5 (las 15 columnas), calcula el R² de validación cruzada de `Ridge` con `alpha` en 0.01, 0.1, 1, 10, 100 y 1000. Grafica R² contra alpha (escala logarítmica). ¿Cuál es el mejor?

**Ejercicio 5 (avanzado). El camino de Lasso.** Para las 15 columnas de la sección 5, entrena `Lasso` con 30 valores de alpha entre 0.001 y 1 (`np.logspace(-3, 0, 30)`) y guarda los coeficientes. Grafica cada coeficiente contra alpha (una línea por característica, eje x logarítmico). ¿En qué orden van llegando a cero? ¿Cuál es la última característica en «sobrevivir»?

## 🏆 Reto opcional: diabetes

scikit-learn trae un conjunto de datos médicos: `load_diabetes()`. Tiene 442 pacientes, 10 mediciones (edad, sexo, índice de masa corporal, presión arterial y seis análisis de sangre) y una medida de cómo avanzó la enfermedad un año después.

1. Compara con validación cruzada (R²) una regresión lineal, Ridge y Lasso, eligiendo alpha con `RidgeCV` y `LassoCV` (de `sklearn.linear_model`).
2. ¿Qué características deja Lasso? ¿Cuál tiene el coeficiente más grande?
3. Agrega `PolynomialFeatures(2)` antes del modelo. ¿Cuántas columnas hay ahora? ¿Mejora la regresión lineal? ¿Y Lasso?

## 🎬 Cierre

Hoy aprendiste que:
- La **regresión múltiple** es un sistema de matrices: una columna por característica más una columna de unos, resuelto con **mínimos cuadrados**.
- Para comparar coeficientes hay que **normalizar**, y las columnas de texto se convierten en números (sí → 1, no → 0).
- Creando características como $v^2$, un modelo **lineal** dibuja **curvas**: la distancia de frenado crece con el cuadrado de la velocidad.
- Demasiadas características llevan al **sobreajuste**, con coeficientes enormes.
- La **regularización** castiga los coeficientes grandes: **Ridge** los encoge, **Lasso** además elige características. La fuerza λ se elige con **validación cruzada**.

**Para pensar:** en ML 5, un solo árbol de decisión sobreajustaba con facilidad. ¿Qué pasaría si entrenáramos **cientos** de árboles distintos y los pusiéramos a **votar**? En la próxima lección veremos los **bosques aleatorios**, donde muchos modelos mediocres juntos forman uno excelente. 🌲🌲🌲